In [1]:
# Install convokit and import dataset and pandas

!pip install convokit
import pandas as pd

from convokit import Corpus, download
corpus = Corpus(filename=download("persuasionforgood-corpus"))
corpus.print_summary_stats()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Created wheel for convokit: filename=convokit-4.1.0-py3-none-any.whl size=292185 sha256=5e26677e6f918d7fb66afba79a00e14d49049755c468df075f86c7af51f57760
  Stored in directory: /root/.cache/pip/wheels/5e/d8/32/b867fe29313972bfba4c

In [2]:
# Creating dataframe so I can download a CSV of the dataset to use in Jamovi. Collecting the metadata.
rows = []

for conversation in corpus.iter_conversations():
    rows.append({
        "dialogue_id":  conversation.meta.get("dialogue_id", None),
        "persuader_id": conversation.meta.get("user_er", None),
        "persuadee_id": conversation.meta.get("user_ee", None),
        "donation_ee":  conversation.meta.get("donation_ee", None),
        "donation_er":  conversation.meta.get("donation_er", None),
        "is_annotated": conversation.meta.get("is_annotated", None),
    })

df_convos = pd.DataFrame(rows)
print(df_convos.shape)
df_convos.head()

(1017, 6)


,dialogue_id,persuader_id,persuadee_id,donation_ee,donation_er,is_annotated
0,20180904-045349_715_live,A3A07QA5U733HQ,A25L985XCNESXE,0.00,0.00,True
1,20180904-154250_98_live,A3GGA28ZY5CYTH,AG3ISFZMFGDQ9,2.00,0.00,False
2,20180904-024226_703_live,A22WWSTT8TU7G1,A2JQOU78XJA6VP,0.05,0.05,True
3,20180904-100019_870_live,A29NICQTS9B05U,A302K8B1H9ISJA,0.00,0.00,False
4,20180904-001208_706_live,A3LQ2F30FKC0CZ,A1RGD548VIVY96,1.00,0.00,False


In [3]:
# getting persuader information. creating new dataframe
rows = []

for speaker in corpus.iter_speakers():
    rows.append({
        "persuader_id":  speaker.id,
        # Big Five Personality Traits
        "extrovert":     speaker.meta.get("extrovert", None),
        "agreeable":     speaker.meta.get("agreeable", None),
        "conscientious": speaker.meta.get("conscientious", None),
        "neurotic":      speaker.meta.get("neurotic", None),
        "open":          speaker.meta.get("open", None),
        # Demographics
        "age":           speaker.meta.get("age", None),
        "sex":           speaker.meta.get("sex", None),
        "race":          speaker.meta.get("race", None),
        "marital":       speaker.meta.get("marital", None),
        "employment":    speaker.meta.get("employment", None),
        "income":        speaker.meta.get("income", None),
        "edu":           speaker.meta.get("edu", None),
        "religion":      speaker.meta.get("religion", None),
        "ideology":      speaker.meta.get("ideology", None),
    })

df_speakers = pd.DataFrame(rows)
print(df_speakers.shape)
df_speakers.head()

(1285, 15)


,persuader_id,extrovert,agreeable,conscientious,neurotic,open,age,sex,race,marital,employment,income,edu,religion,ideology
0,A3A07QA5U733HQ,3.2,3.2,3.6,1.6,3.6,34.0,Male,White,Unmarried,Employed for wages,5.0,Less than four-year college,Other religion,Liberal
1,A25L985XCNESXE,3.2,4.0,3.8,2.0,3.2,50.0,Female,White,Married,Employed for wages,10.0,Less than four-year college,Protestant,Conservative
2,A3GGA28ZY5CYTH,3.6,3.8,4.0,4.0,3.6,25.0,Female,White,Married,Other,6.0,Four-year college,Atheist,Moderate
3,AG3ISFZMFGDQ9,3.0,3.6,3.2,3.0,3.0,30.0,Male,White,Unmarried,Employed for wages,2.0,Less than four-year college,Atheist,Liberal
4,A22WWSTT8TU7G1,1.0,5.0,2.8,1.0,4.6,36.0,Female,White,Unmarried,Employed for wages,1.0,Less than four-year college,Catholic,Moderate


In [4]:
#Merging dataframes of persuader ID and information
df_merged = pd.merge(df_convos, df_speakers, on="persuader_id", how="left")

#defining which are numerical columns and categorical columns
numeric_cols = ["donation_ee", "donation_er", "extrovert", "agreeable",
                "conscientious", "neurotic", "open", "age", "income"]

categorical_cols = ["sex", "race", "marital", "employment", "edu", "religion", "ideology"]

#Average the numeric columns per persuader - making sure the same participants is not counted as two people which could skew results.
df_numeric = df_merged.groupby("persuader_id")[numeric_cols].mean().reset_index()

#use first value in categorical columns
df_categorical = df_merged.groupby("persuader_id")[categorical_cols].first().reset_index()

#use persuader ID as the axis to merge everything back together
df_persuaders = pd.merge(df_numeric, df_categorical, on="persuader_id")

#create final dataframe
print(df_persuaders.shape)
print(df_persuaders.columns.tolist())
df_persuaders.head()

(701, 17)
['persuader_id', 'donation_ee', 'donation_er', 'extrovert', 'agreeable', 'conscientious', 'neurotic', 'open', 'age', 'income', 'sex', 'race', 'marital', 'employment', 'edu', 'religion', 'ideology']


,persuader_id,donation_ee,donation_er,extrovert,agreeable,conscientious,neurotic,open,age,income,sex,race,marital,employment,edu,religion,ideology
0,A03172343NB8PHOPICC1,0.0,0.05,3.8,4.0,3.8,2.4,4.6,22.0,6.0,Male,White,Unmarried,Employed for wages,Four-year college,Other religion,Conservative
1,A08663392NYK5WHK2M2M8,0.0,0.00,1.2,1.2,4.4,2.8,3.0,34.0,8.0,Male,White,Married,Employed for wages,Less than four-year college,Other religion,Conservative
2,A10249252O9I20MRSOBVF,0.0,0.00,3.0,4.2,4.8,1.0,3.4,54.0,6.0,Female,White,Married,Employed for wages,Less than four-year college,Catholic,Liberal
3,A10ARSSXF5SFCW,0.0,1.00,3.8,4.4,4.6,1.2,4.2,27.0,5.0,Female,Other,Married,Employed for wages,Less than four-year college,Protestant,Liberal
4,A10F0CHSNURXF5,0.1,0.05,5.0,4.6,4.8,1.2,4.8,46.0,12.0,Female,White,Married,Employed for wages,Four-year college,Catholic,Conservative


In [5]:
# drop any rows that do not contain a value in the bfp traits
df_persuaders = df_persuaders.dropna(subset=["extrovert", "agreeable", "conscientious", "neurotic", "open"])
# Add binary donated column
df_persuaders["donated"] = (df_persuaders["donation_ee"] > 0).astype(int)

print(df_persuaders.shape)
print(df_persuaders["donated"].value_counts())

(699, 18)
donated
1    411
0    288
Name: count, dtype: int64


In [6]:
#dataframe to csv
df_persuaders.to_csv("persuasion_for_good_clean.csv", encoding='utf-8', index=False)

In [7]:
#download CSV to work in Jamovi
from google.colab import files
files.download("persuasion_for_good_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>